In [ ]:
import hats_import
import hats
import pandas as pd
import nested_pandas as npd

In [ ]:
(hats_import.__version__, hats.__version__)


## Photo-Z import for DP2

## Creating the import file list

In [ ]:
#paths
pz = "/sdf/home/z/ztq1996/u/photoz/dp2/production_lsdb/data"
band4 = "/sdf/home/z/ztq1996/u/photoz/dp2/production_lsdb/data/4band" # dir of .pq
band6 = "/sdf/home/z/ztq1996/u/photoz/dp2/production_lsdb/data/6band" # dir of .pq
tract_csv = "/sdf/home/z/ztq1996/u/photoz/dp2/production_lsdb/data/tract_lists.csv"

In [ ]:
# tract list details which band set is recommended for each tract.
tract_df = pd.read_csv(tract_csv)
tract_df

In [ ]:
# create the filenames for each tract based on the recommended band
fnames = tract_df.apply(lambda x: 
                f"{pz}/{x.recommended_band}band/pz_point_estimates_tract_{x.tract}.parquet",
                axis=1)
tract_with_fnames = tract_df.join(fnames.rename("filenames"))
tract_with_fnames

In [ ]:
# verify we can read a file
npd.read_parquet(tract_with_fnames.loc[52]["filenames"])

In [ ]:
# verify recommendation consistency
print(len(tract_with_fnames.query("filenames.str.contains('6band')")) == len(tract_with_fnames.query("recommended_band == 6")))
print(len(tract_with_fnames.query("filenames.str.contains('4band')")) == len(tract_with_fnames.query("recommended_band == 4")))

In [ ]:
# Collect the file list
file_list = tract_with_fnames.filenames.tolist()
file_list[0:5]

## Importing

In [ ]:
from hats_import.catalog.arguments import ImportArguments
from dask.distributed import Client
from hats_import.pipeline import pipeline_with_client

args = ImportArguments(
    sort_columns="objectId",
    ra_column="ra",
    dec_column="dec",
    input_file_list=file_list,
    file_reader="parquet",
    output_artifact_name="photo_z_point_estimates",
    output_path="/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2_pz",
)

with Client(n_workers=8) as client:
    print(client)
    pipeline_with_client(args, client)

## Verification

In [ ]:
import lsdb

pz = lsdb.open_catalog("/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2_pz/photo_z_point_estimates")
pz

In [ ]:
len(pz)

In [ ]:
pz.plot_pixels()